# PandasDataFrameOutputParser

**Pandas DataFrame**은 Python 프로그래밍 언어에서 널리 사용되는 데이터 구조로, 데이터 조작 및 분석을 위한 강력한 도구입니다. DataFrame은 구조화된 데이터를 효과적으로 다루기 위한 포괄적인 도구 세트를 제공하며, 이를 통해 데이터 정제, 변환 및 분석과 같은 다양한 작업을 수행할 수 있습니다.

이 **출력 파서**는 사용자가 임의의 Pandas DataFrame을 지정하여 해당 DataFrame에서 데이터를 추출하고, 이를 형식화된 사전(dictionary) 형태로 조회할 수 있게 해주는 LLM(Large Language Model) 기반 도구입니다.


In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
# !pip install pandas

import pprint / from typing import Any, Dict
둘 다 Python 표준 라이브러리입니다.

- pprint (pretty print): 딕셔너리나 리스트처럼 중첩된 자료를 줄바꿈·들여쓰기해서 보기 좋게 출력합니다. 이 노트북에서는 파서 결과(딕셔너리)를 pprint.pprint(...)로 찍는 데 씁니다.

- print({'a': [1,2,3], 'b': {'x': 1}})          # 한 줄로 찍힘
pprint.pprint({'a': [1,2,3], 'b': {'x': 1}})  # 구조대로 정렬돼서 찍힘

typing.Any, Dict: 타입 힌트용입니다. 
아래쪽에 def format_parser_output(parser_output: Dict[str, Any]) -> None: 같은 함수가 있을 텐데, "문자열 키에 아무 값이나 담긴 딕셔너리를 받는다"는 표시일 뿐 실행에는 영향이 없습니다. 최신 Python에서는 dict[str, Any]로 써도 됩니다.

In [ ]:
import pprint
from typing import Any, Dict

import pandas as pd
from langchain_classic.output_parsers import PandasDataFrameOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

이 venv는 langchain 1.4.0 / langchain-core 1.6.2입니다. 

LangChain 1.0에서 핵심 패키지를 대폭 정리하면서, 교재(langchain-kr, 0.x 기준)에 나오는 langchain.output_parsers.*의 레거시 파서들(PandasDataFrameOutputParser, DatetimeOutputParser, EnumOutputParser, StructuredOutputParser, OutputFixingParser 등)은 langchain-classic으로 이동했습니다. 

langchain_core에는 PydanticOutputParser, JsonOutputParser, CommaSeparatedListOutputParser, StrOutputParser 같은 기본 파서만 남아 있습니다.

In [9]:
# # ChatOpenAI 모델 초기화 (gpt-3.5-turbo 모델 사용을 권장합니다)
# model = ChatOpenAI(temperature=0, model="gpt-3.5-turbo")

model = ChatOpenAI(temperature=0, model="gpt-4o-mini")

`format_parser_output` 함수는 파서 출력을 사전 형식으로 변환하고 출력 형식을 지정하는 데 사용됩니다. 

In [10]:
# 출력 목적으로만 사용
def format_parser_output(parser_ourput: Dict[str, Any]) -> None:
    # 파서 출력의 키들을 순회
    for key in parser_ourput.keys():
        # 각 키의 값을 딕셔너리로 변환
        parser_ourput[key] = parser_ourput[key].to_dict()
    # 정돈된 상태로 출력
    return pprint.PrettyPrinter(width=4, compact=True).pprint(parser_ourput)

- `titanic.csv` 데이터를 읽어온 뒤 DataFrame 을 로드하여 `df` 변수에 할당합니다.
- PandasDataFrameOutputParser를 사용하여 DataFrame을 파싱합니다.

In [20]:
import os
print(os.getcwd())        # 지금 "."이 어디인지
print(os.listdir("."))    # 그 폴더에 뭐가 있는지 → titanic.csv 보이면 파일명만 쓰면 됨

d:\hykim\hanwha_0902\01_class\week03\0916
['ch6_01_pydantic_ouput_parser.ipynb', 'ch6_02_comma_separated_list_output_parser.ipynb', 'ch6_03_structured_output_parser.ipynb', 'ch6_04_json_output_parser.ipynb', 'ch6_05_pandas_data_frame_output_parser.ipynb', 'pandas_class.ipynb', 'pandas_class.py', 'titanic.csv']


df = pd.read_csv("titanic.csv") #-> ✅  ("./titanic.csv"도 동일)
실행 파이썬 파일과 같은 폴더내에 있을 때는 파일명만 적어도 가능


In [21]:
# 원하는 Pandas DataFrame을 정의합니다.
df = pd.read_csv("titanic.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [22]:
# 파서를 설정하고 프롬프트 템플릿에 지시사항을 주입합니다.
parser = PandasDataFrameOutputParser(dataframe=df)

# 파서의 지시사항을 출력합니다.
print(parser.get_format_instructions())

The output should be formatted as a string as the operation, followed by a colon, followed by the column or row to be queried on, followed by optional array parameters.
1. The column names are limited to the possible columns below.
2. Arrays must either be a comma-separated list of numbers formatted as [1,3,5], or it must be in range of numbers formatted as [0..4].
3. Remember that arrays are optional and not necessarily required.
4. If the column is not in the possible columns or the operation is not a valid Pandas DataFrame operation, return why it is invalid as a sentence starting with either "Invalid column" or "Invalid operation".

As an example, for the formats:
1. String "column:num_legs" is a well-formatted instance which gets the column num_legs, where num_legs is a possible column.
2. String "row:1" is a well-formatted instance which gets row 1.
3. String "column:num_legs[1,2]" is a well-formatted instance which gets the column num_legs for rows 1 and 2, where num_legs is a p

컬럼에 대한 값을 조회하는 예제입니다.

In [ ]:
# 열 작업 예시
df_query = "Age Column을 조회해 주세요."

# 프롬프트 템플릿 설정
prompt = PromptTemplate(
    template="Answer the user query,\n{format_instructions}\n{question}\n",
    input_variables=["question"], #입력 변수 설정
    partial_variables={
        "format_instructions": parser.get_format_instructions()
    },  # 부분 변수 설정
)

# 체인 생성
chain = prompt | model | parser

# 체인 실행
parser_output = chain.invoke({"question": df_query})

# 출력
format_parser_output(parser_output)


OutputParserException: Unsupported request type '"column'.                         Please check the format instructions.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 